# SENTINEL — Chẩn đoán luồng A (đặc trưng hình ảnh) cho hành vi đánh nhau

`visual_fight_v1` đạt AUC theo frame 0,872 trên UBI-Fights nhưng AUC trong từng video đánh nhau chỉ 0,495, thấp hơn đường cơ sở chuyển động (0,562). Khi train, loss xếp hạng về 0,002 còn AUC trong video ở validation giảm từ 0,50 xuống 0,41: model học thuộc cảnh. Notebook này trả lời câu hỏi quyết định bước tiếp theo: **đặc trưng S3D đã trích có tín hiệu đánh nhau không?**

| Cách | Ý nghĩa |
|---|---|
| `motion` | Đường cơ sở năng lượng chuyển động |
| `zero_shot` | Xác suất các lớp đánh nhau có sẵn của Kinetics (đấm, vật, tát, húc đầu, đá) — không train, không thể học thuộc |
| `linear`, `temporal` | Linear probe và đầu dự đoán nhỏ (64 đơn vị, dropout 0,5) |
| `*_scene` | Trừ trung bình các clip trước đó của cùng camera (bỏ bối cảnh; nhân quả, chạy thật được) |
| `*_videomean` | Trừ trung bình cả video (không nhân quả; chỉ để chẩn đoán) |

Các probe chọn epoch theo AUC trong video trên validation (cùng cách chia với lần train). **Chỉ đọc đặc trưng đã trích** (`features/s3d_v1`), không cần giải nén video: nhãn UBI-Fights được lấy thẳng từ file zip trên Drive. Khoảng 15–30 phút; GPU nào cũng được.

**Đọc kết quả (mục 6, dòng `KẾT LUẬN` cũng ghi vào `artifacts/visual_fight_v1/run_log.txt`):**
- `SIGNAL`: có cách vượt đường cơ sở chuyển động ≥ 0,05 trong video → đặc trưng có tín hiệu, train lại với cấu hình đó (thêm RLVS).
- `NO SIGNAL BEYOND MOTION`: S3D toàn khung hình không thấy đánh nhau → trích lại đặc trưng trên vùng cắt quanh nhóm người.

## 0. Cấu hình

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/TranDuon/anomaly-behavior-detector-system.git"
BRANCH = "main"
DRIVE_ROOT = Path("/content/drive/MyDrive/SENTINEL")
REPO_DIR = DRIVE_ROOT / "repo" / "anomaly-behavior-detector-system"
ZIP_DIR = DRIVE_ROOT / "datasets" / "fight"           # UBI_FIGHTS.zip đã lưu ở lần train trước
FEATURE_ROOT = DRIVE_ROOT / "features" / "s3d_v1"      # đặc trưng của visual_fight_v1
ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / "visual_fight_v1"
OUTPUT_DIR = ARTIFACT_DIR / "diagnostics"
DEDUP_DIR = DRIVE_ROOT / "artifacts" / "fight" / "dedup"
LOCAL_DATA = Path("/content/datasets")
UBI_ROOT = LOCAL_DATA / "UBI_FIGHTS"
SCF_ROOT = LOCAL_DATA / "fight-detection-surv-dataset"
SCF_URL = "https://github.com/sayibet/fight-detection-surv-dataset.git"
AUTO_DISCONNECT = True         # chạy xong hoặc gặp lỗi thì ghi log, đồng bộ Drive rồi ngắt runtime
print("Features:", FEATURE_ROOT, "| Kết quả:", OUTPUT_DIR)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for path in [REPO_DIR.parent, ZIP_DIR, FEATURE_ROOT, ARTIFACT_DIR, LOCAL_DATA]:
    path.mkdir(parents=True, exist_ok=True)

import time
from contextlib import contextmanager
RUN_LOG = ARTIFACT_DIR / "run_log.txt"

def note(message):
    # Ghi một dòng vào log trên Drive (không ngắt runtime).
    with open(RUN_LOG, "a", encoding="utf-8") as log:
        log.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} {message}\n")
    print(message)

def finish_and_disconnect(message):
    # Ghi log lên Drive; khi AUTO_DISCONNECT thì đồng bộ Drive rồi ngắt runtime để dừng tốn compute unit.
    note(message)
    if AUTO_DISCONNECT:
        from google.colab import runtime
        drive.flush_and_unmount()
        runtime.unassign()

@contextmanager
def step(name):
    try:
        yield
    except Exception as error:
        finish_and_disconnect(f"LỖI ở mục {name}: {error!r} — xem output của mục này, sửa rồi chạy lại.")
        raise

print("Log chạy:", RUN_LOG, "| AUTO_DISCONNECT =", AUTO_DISCONNECT)

## 2. Đồng bộ code mới nhất

In [ ]:
with step('2. Đồng bộ code'):
    import os, subprocess

    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["git", "log", "--oneline", "-1"], check=True)

## 3. Cài dependency

In [ ]:
with step('3. Cài dependency'):
    import sys, torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("PyTorch", torch.__version__, "| device:", DEVICE)
    assert any(FEATURE_ROOT.glob("ubi/*.npz")), f"Không có đặc trưng trong {FEATURE_ROOT}: chạy notebook train_visual_fight_drive trước"

## 4. Nhãn dataset (không giải nén video)
Từ `UBI_FIGHTS.zip` trên Drive chỉ lấy file nhãn và `test_videos.csv`; video được thay bằng file rỗng cùng tên (chỉ cần tên, đặc trưng đã có). Surveillance Camera Fight clone lại từ GitHub (~90 MB).

In [ ]:
with step('4. Nhãn dataset'):
    import zipfile
    ubi_zip = ZIP_DIR / "UBI_FIGHTS.zip"
    assert ubi_zip.exists(), f"Không thấy {ubi_zip}"
    if not (UBI_ROOT / "test_videos.csv").exists():
        with zipfile.ZipFile(ubi_zip) as archive:
            for name in archive.namelist():
                if name.endswith("/"):
                    continue
                if name.lower().endswith(".mp4"):
                    target = LOCAL_DATA / name
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.touch()
                elif "/annotation/" in name or name.endswith("test_videos.csv"):
                    archive.extract(name, LOCAL_DATA)
    print("UBI-Fights:", len(list(UBI_ROOT.glob("videos/*/*.mp4"))), "video,",
          len(list(UBI_ROOT.glob("annotation/*.csv"))), "file nhãn")
    if not SCF_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", SCF_URL, str(SCF_ROOT)], check=True)
    print("Surveillance Camera Fight:", len(list(SCF_ROOT.rglob("*.mp4"))), "clip")

## 5. Chẩn đoán

In [ ]:
with step('5. Chẩn đoán'):
    command = [sys.executable, "-u", "scripts/diagnose_visual_fight.py",
               "--dataset", f"ubi:{UBI_ROOT}", "--dataset", f"scf:{SCF_ROOT}",
               "--feature-root", str(FEATURE_ROOT), "--output-dir", str(OUTPUT_DIR),
               "--exclude", str(DEDUP_DIR / "exclude.json"), "--device", DEVICE]
    subprocess.run(command, cwd=REPO_DIR, check=True)

## 6. Kết quả
Cột chính: **test trong video** (AUC trung vị trong 40 video đánh nhau của tập test). Lần train `visual_fight_v1`: 0,495; luồng B: 0,537.

In [ ]:
with step('6. Kết quả'):
    import json
    report = json.loads((OUTPUT_DIR / "visual_diagnostics.json").read_text())
    header = f"{'cách':20s} {'val trong video':>16s} {'test trong video':>17s} {'test frame':>11s} {'test short':>11s} {'SCF clip':>9s}"
    print(header); print("-" * len(header))
    for name, row in report["results"].items():
        scf = row.get("scf_clip_auc")
        print(f"{name:20s} {row['val_video_auc_median']:16.3f} {row['test_video_auc_median']:17.3f} "
              f"{row['test_frame_auc']:11.3f} {row['test_short_frame_auc']:11.3f} {'' if scf is None else f'{scf:9.3f}'}")
    print("\nTốt nhất trên validation:", report["best_on_validation"])
    note(f"KẾT LUẬN chẩn đoán luồng A: {report['verdict']}")

## 7. Kết thúc: ngắt runtime

In [ ]:
finish_and_disconnect(f"Hoàn tất chẩn đoán. Kết quả: {OUTPUT_DIR / 'visual_diagnostics.json'}")